# 데이터 수집
사용법 : 1~9은 건들지 않고 항목 **10. 데이터 수집 실행**만 사용. 

In [1]:
import pandas as pd
import numpy as np
import requests, time, re, os, random
from bs4 import BeautifulSoup

# 1-1. API 주소 및 요청 설정

In [2]:
API_KEY = "B1FC36C7D790B5F17DCD8E33F5C33DF2"

# TEST ID
TEST_APP_ID = 1245620 # 게임 엘든링
TEST_STEAM_ID = 76561199824606664

API_URL = {
    "OWNED_GAMES" : "https://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/",
    "RECENT_GAMES" : " https://api.steampowered.com/IPlayerService/GetRecentlyPlayedGames/v0001/",
    "APP_DETAILS" : "https://store.steampowered.com/api/appdetails",
    "STORE_PAGE" : "https://store.steampowered.com/app/",
    "REVIEWS" : "https://store.steampowered.com/appreviews/"
}

session = requests.Session()

def change_user_agent() :
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/151.0.0.0 Safari/537.36"
        )
    })

change_user_agent()

# 1-2. API 요청 지연 및 재시도

In [3]:
def rate_limited_request(url, params=None, timeout=15, max_retries=5, base_delay=0.3):
    for attempt in range(max_retries):
        response = session.get(url, params=params)
        if response.status_code == 429:
            wait = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
            print(f"429 발생으로 {wait:.1f}초 대기 후 재시도 ({attempt+1}/{max_retries})")
            time.sleep(wait)
            continue
        response.raise_for_status()
        time.sleep(base_delay) # 기본 딜레이
        return response
    raise RuntimeError(f"최대 재시도 초과: {url}")

# 2. 게임 정보 데이터 수집
- 장르
- 카테고리
- (게임 페이지 기준) 추천 개수, 메타크리틱 점수, 출시일, 가격

In [4]:
game_info_cache = {}

def get_game_info(appid):
    if appid in game_info_cache:
        cache = game_info_cache[appid]
        return cache["genre"], cache["categories"], cache["info_tup"]

    params = {"appids": appid, "l": "english"}

    try:
        response = rate_limited_request(API_URL["APP_DETAILS"], params=params) #session.get(API_URL["APP_DETAILS"], params=params, timeout=15)
        response_status = response.status_code
        if response_status != 200: 
            print("failed", response_status)
            return [],[],(0, "Unknown","Unknown","Unknown")

        data = response.json()
        
        app_data = data.get(str(appid), {})
        if not app_data.get("success"): return [],[],(0, "Unknown","Unknown","Unknown")
        game_info = app_data.get("data", {})
        """
        print(game_info.keys())
        print("recommendations", game_info.get("recommendations"))
        print("release_date", game_info.get("release_date"))
        print("price_overview", game_info.get("price_overview"))
        print("metacritic", game_info.get("metacritic"))
        """

        # 장르 정보
        genres = game_info.get("genres", [])
        genre_list = [genre.get("description") for genre in genres]

        # 카테고리 정보
        categories = game_info.get("categories", [])
        category_list = [category.get("description") for category in categories]

        # 기타 정보 종합
        recommendations = game_info.get("recommendations", {}).get("total", 0)
        release_date = game_info.get("release_date", {}).get("date", "Unknown")
        metacritic = game_info.get("metacritic", {}).get("score", "Unknown")

        # 가격 정보
        if game_info.get("is_free"): price = 0 
        else: price = game_info.get("price_overview", {}).get("final", "Unknown")
        if price != "Unknown" : price /= 100
        info_tup = (recommendations, metacritic, release_date, price)

        #print(game_info.get("price_overview"))
        
        game_info_cache[appid] = {"genre":genre_list, "categories":category_list, "info_tup":info_tup}
        return genre_list, category_list, info_tup
        

    except Exception as e:
        print(f"AppID {appid} 상세정보오류:", e)
        return [],[],(0, "Unknown","Unknown","Unknown")

print(get_game_info(TEST_APP_ID))

(['Action', 'RPG'], ['Single-player', 'Multi-player', 'PvP', 'Online PvP', 'Co-op', 'Online Co-op', 'Steam Achievements', 'Full controller support', 'Steam Trading Cards', 'Camera Comfort', 'Custom Volume Controls', 'Playable without Timed Input', 'Save Anytime', 'Stereo Sound', 'Surround Sound', 'Steam Cloud', 'Family Sharing'], (827619, 94, '24 Feb, 2022', 59.99))


# 3. 게임 리뷰 데이터 수집
- 전체 리뷰 수, 긍정 리뷰 수, 긍정 리뷰 수, 리뷰 수 비율 레이팅

In [5]:
game_review_info_cache = {}

def get_game_review_info(appid):
    if appid in game_review_info_cache:
        cache = game_review_info_cache[appid]
        return cache
        
    URL = API_URL["REVIEWS"] + str(appid)
    params = {"json":1, "language":"all", "filter":"all", "review_type":"all","purchase_type":"all"}
    try:
        response = rate_limited_request(URL, params=params) # session.get(URL, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        
    except Exception as e:
        print(f"AppID {appid} 리뷰 오류:", e)
        game_review_info_cache[appid] = ("Unknown", "Unknown", "Unknown", "Unknown")
        return ("Unknown", "Unknown", "Unknown", "Unknown")
        
    summary = data.get("query_summary", {})
    total_reviews = summary.get("total_reviews", "Unknown")
    positive_reviews = summary.get("total_positive", "Unknown")
    negative_reviews = summary.get("total_negative", "Unknown")
    if positive_reviews == "Unknown" or negative_reviews == "Unknown" or positive_reviews == None or negative_reviews == None : 
        rating_by_reviews = "Unknown"
    if total_reviews == 0 : rating_by_reviews = 0
    else : rating_by_reviews = round(positive_reviews / (positive_reviews + negative_reviews) * 100, 2)

    game_review_info_cache[appid] = (total_reviews, positive_reviews, negative_reviews, rating_by_reviews)
    return (total_reviews, positive_reviews, negative_reviews, rating_by_reviews)

print(get_game_review_info(TEST_APP_ID))

(1149229, 1069705, 79524, 93.08)


# 4. Steam Store 사용자 태그 데이터 수집

In [6]:
TAG_PATTERNS = [r'class="app_tag"[^>]*>\s*([^<]+)',  \
                r'class="app_tag[^"]*"[^>]*>\s*([^<]+)', \
                r'data-tagid="[^"]+"[^>]*>\s*([^<]+)']

game_tags_cache = {}

def get_steam_tags(appid):
    if appid in game_tags_cache:
        return game_tags_cache[appid]
    
    URL = f"{API_URL['STORE_PAGE']}{appid}/"

    try:
        response = rate_limited_request(URL) # session.get(URL, timeout=15)

        response.raise_for_status()
        html = response.text

    except Exception as e:
        print(f"AppID {appid} 태그 오류:", e)
        game_tags_cache[appid] = []
        return []

    # 방법 1 : 태그가 들어어있는 요소들 우선 탐색
    soup = BeautifulSoup(html, "html.parser")
    tags = []
    tag_elements = soup.select(".app_tag")

    for element in tag_elements:
        tag = element.get_text(strip=True)
            
        if not tag: continue
        if tag == "+": continue

        if tag not in tags: tags.append(tag)

    # 방법 2 (방법 1 실패 시) : 보조 탐색
    if not tags:
        for pattern in TAG_PATTERNS:
            matches = re.findall(pattern, html, flags=re.IGNORECASE)
            
            for match in matches:
                tag = (match.replace("&amp;", "&").strip())
                if tag and tag != "+" and tag not in tags :
                    tags.append(tag)

    # 태그 정리
    cleaned_tags = []

    for tag in tags:
        tag = tag.replace("\n", "").replace("\r", "").strip()
        if tag and tag != "+" and tag not in cleaned_tags :
            cleaned_tags.append(tag)

    game_tags_cache[appid] = cleaned_tags
    return cleaned_tags

print(get_steam_tags(TEST_APP_ID))

['Souls-like', 'Open World', 'Dark Fantasy', 'RPG', 'Difficult', 'Action RPG', 'Multiplayer', 'Third Person', 'Fantasy', 'Singleplayer', 'Online Co-Op', 'Action', 'Co-op', 'Atmospheric', 'Great Soundtrack', 'PvP', 'Violent', '3D', 'Character Customization', 'Family Friendly']


# 5. 사용자 보유 게임 데이터 수집

In [7]:
def get_owned_games_of_user(steam_id, max_games=False) :
    params = {"key":API_KEY, "steamid":steam_id, "include_appinfo":True, \
              "include_played_free_games":True, "format":"json"}

    try:
        response = rate_limited_request(API_URL["OWNED_GAMES"], params=params) # session.get(API_URL["OWNED_GAMES"], params=params, timeout=15)
        
        response.raise_for_status()
        data = response.json()

        games_count = data.get("response", {}).get("game_count", [])
        games = data.get("response", {}).get("games", [])
        # print(games[0].keys())

    except Exception as e:
        print(f"steam_id {steam_id} 보유 게임 오류:", e)
        return []

    if not games :
        print(f"steam_id {steam_id} 보유 게임 없음")
        return []

    # 상위 max_games 개 게임만 수집
    if max_games and games_count > max_games:
        games = sorted(games, key=lambda g: g.get("playtime_forever", 0), reverse=True)[:max_games]
    
    res = []
    for game in games :
        appid = game.get("appid")
        game_name = game.get("name", "Unknown")
        playtime_mins = game.get("playtime_forever", 0)
        res.append((appid, game_name, playtime_mins))
    return res

print(get_owned_games_of_user(TEST_STEAM_ID))

[(578080, 'PUBG: BATTLEGROUNDS', 139), (977950, 'A Dance of Fire and Ice', 459), (1245620, 'ELDEN RING', 575), (2250500, 'Rhythia', 378)]


# 6. 사용자 최근 2주 플레이 데이터 수집

In [8]:
def get_recent_games_of_user(steam_id) :
    params = {"key": API_KEY, "steamid":steam_id, "foramt":"json"}

    try :
        response = rate_limited_request(API_URL["RECENT_GAMES"], params=params) # session.get(API_URL["RECENT_GAMES"], params=params, timeout=15)

        response.raise_for_status()
        data = response.json()
        games = data.get("response", {}).get("games", [])

    except Exception as e:
        print(f"steam_id {steam_id} 최근 게임 오류:", e)
        return []

    if not games:
        print(f"steam_id {steam_id} 최근 게임 없음")
        return []

    res = []
    for game in games :
        appid = game.get("appid")
        game_name = game.get("name", "Unknown")
        playtime_mins = game.get("playtime_2weeks", 0)
        res.append((appid, game_name, playtime_mins))
    return res

print(get_recent_games_of_user(TEST_STEAM_ID))

[(1245620, 'ELDEN RING', 89), (977950, 'A Dance of Fire and Ice', 33)]


# 7. Steam Store 게임 리뷰에서 사용자 ID 데이터 수집

In [9]:
def get_reviews(appid, cursor="*", num_per_page=100, only_positive=True):
    URL = API_URL["REVIEWS"] + str(appid)
    review_type = "positive" if only_positive else "all"
    params = {"json":1, "filter":"updated", "language":"all", "cursor":cursor, "num_per_page":num_per_page, \
        "review_type":review_type, "purchase_type":"steam"}

    try:
        response = rate_limited_request(URL, params=params) # session.get(URL, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

    except Exception as e:
        print(f"AppId {appid} 리뷰 오류:", e)
        return []

    if not data :
        print(f"AppId {appid} 리뷰 없음")

    return data

def get_user_ids_from_app(appid, id_nums, only_positive=True, minimum_playtime_mins=0, minimum_games=0, max_games=False):
    id_set = set()
    cursor = "*"
    while len(id_set) < id_nums :
        # 리뷰 데이터 100개 수집
        data = get_reviews(appid, cursor=cursor, num_per_page=100, only_positive=only_positive)
        if not data :
            print("리뷰 로딩 중 오류")
            break
        
        reviews = data.get("reviews", [])
        if not reviews : break

        for review in reviews:
            if len(id_set) >= id_nums : break
            
            user = review.get("author", {})
            if user.get("playtime_forever", 0) < minimum_playtime_mins : continue
            elif user.get("num_games_owned", 0) < minimum_games : continue
            elif max_games and user.get("num_games_owned", 0) > max_games : continue

            id = user.get("steamid")
            if id : id_set.add(id)

        cursor = data.get("cursor")
        if not cursor: break

        time.sleep(3)
    return list(id_set)

print("return example : get_user_ids_from_app()")
TEST_APP_ID = 1245620 # 게임 엘든링
test_ids_list = get_user_ids_from_app(TEST_APP_ID, 15, minimum_playtime_mins=600, minimum_games=15)
print(test_ids_list)

return example : get_user_ids_from_app()
['76561199177732895', '76561198091122227', '76561199194402331', '76561198166226287', '76561199015330455', '76561198079488752', '76561198885046549', '76561198984355584', '76561198073067799', '76561199835282867', '76561199110612232', '76561198272738490', '76561199065212767', '76561198010613920', '76561198725405351']


# 8. 앱ID 리스트에서 DataFrame 생성

In [10]:
def create_df_from_appid_list(appid_list, ids_per_game, minimum_playtime_mins=600, minimum_games=15, max_search_games=False, max_filter_games=30) :

    data = [] # DataFrame으로 만들 데이터
    
    # 게임 목록 순회
    for app_idx, appid in enumerate(appid_list) :
        user_ids = get_user_ids_from_app(appid, ids_per_game, minimum_playtime_mins=minimum_playtime_mins, \
                                          minimum_games=minimum_games, max_games=max_search_games)
        # 게임에서 찾은 유저 순회
        for id_idx, user_id in enumerate(user_ids) :
            owned_games = get_owned_games_of_user(user_id, max_games=max_filter_games)
            print(f"- {app_idx}번 게임의 {id_idx}번 유저의 보유 게임 {len(owned_games)}개 데이터 수집")

            # 유저의 최근(2주) 게임 순회
            recent_games = get_recent_games_of_user(user_id)
            recent_games_dict = {game_id : playtime for game_id, game_name, playtime in recent_games}
            
            # 유저의 보유 게임 순회
            for game_idx, game_info in enumerate(owned_games):
                game_id, game_name, game_playtime_mins = game_info
                game_playtime_hours = round(game_playtime_mins / 60, 2)

                # 게임 장르, 카테고리, 기타 정보
                genre_list, category_list, info_tup = get_game_info(game_id)
                recommendations, metacritic, release_date, price = info_tup

                # 태그 정보 수집
                tag_list = get_steam_tags(game_id)

                # 리뷰 정보 수집
                tot_rev, pos_rev, neg_rev, rating_by_rev = get_game_review_info(game_id)

                # 해당 게임 최근 플레이 이력 확인
                recent_playtime_mins = recent_games_dict.get(game_id, 0)
                recent_playtime_hours = round(recent_playtime_mins / 60, 2)
                
                print(f"- - {game_idx}. : {game_name}")
                data.append({"steamid":user_id, "appid":game_id, "game_name":game_name, "playtime_hours":game_playtime_hours, \
                             "recent_playtime_hours":recent_playtime_hours, \
                            "genre":", ".join(genre_list) if genre_list else "Unknown", \
                            "tags":", ".join(tag_list) if tag_list else "Unknown", \
                            "steam_categories":", ".join(category_list) if category_list else "Unknown", \
                             "total_reviews":tot_rev, "positive_reviews":pos_rev, "negative_reviews":neg_rev, "rating_by_reviews":rating_by_rev, \
                           "recommendations":recommendations, "metacritic":metacritic, "release_date":release_date, "price(원)":price})
            
    data_df = pd.DataFrame(data)
    data_df.head(10)
    return data_df

TEST_APPID_LIST = ["1245620", "1091500"] # 게임 2개 : 엘든링, 사이버펑크
#test_df = create_df_from_appid_list(TEST_APPID_LIST, 3, max_filter_games=5)
#test_df.head(5)

# 9. DataFrame을 CSV로 저장

In [11]:
FILE_NAME = "steam_user_games_"

def make_csv(data_df, file_num) :
    file_name = FILE_NAME + str(file_num) + ".csv"
    data_df.to_csv(file_name, index=False, encoding="utf-8-sig")

# 10. 데이터 수집 실행

In [12]:
# 사용자 정보를 수집할 게임의 id 목록 입력
# 예시) APPID_LIST = ["1245620", "1091500"] # 게임 2개 : 엘든링, 사이버펑크
APPID_LIST = ["1245620", "1091500", "1086940", "292030", "1623730", "582010", "377160", "489830", "990080", "2246340"]

# 게임 하나 당 수집할 id 개수 입력
IDS_PER_GAME = 15

# CSV 파일 번호 입력
FILE_NUM = 3

# DataFrame 생성
data_df = create_df_from_appid_list(APPID_LIST, IDS_PER_GAME, minimum_playtime_mins=600, \
                                    minimum_games=50, max_search_games=False, max_filter_games=30)
# 파라미터
# minimum_playtime_mins : 어떤 게임에서 사용자 id를 수집할 때, 그 게임의 최소 플레이 시간(분)
# minimum_games : 사용자 id를 수집할 때, 최소 보유 게임 개수
# max_search_games : 사용자 id를 수집할 때, 최대 보유 게임 개수. False로 지정시 상한 지정 안 함
# max_filter_games : 어떤 사용자의 게임 정보를 수집할 때, 보유 게임 중 플레이타임 상위 n개의 데이터만 가져옴. 이때의 n을 지정.

make_csv(data_df, FILE_NUM)

- 0번 게임의 0번 유저의 보유 게임 30개 데이터 수집
- - 0. : ELDEN RING
- - 1. : Grand Theft Auto V Legacy
- - 2. : Bloons TD 6
- - 3. : Apex Legends
- - 4. : ARC Raiders
- - 5. : 7 Days to Die
- - 6. : BeamNG.drive
- - 7. : Blade & Sorcery
- - 8. : Marvel’s Spider-Man Remastered
- - 9. : Red Dead Redemption 2
- - 10. : Sekiro™: Shadows Die Twice
- - 11. : ROUNDS
- - 12. : Garry's Mod
- - 13. : Tom Clancy's Rainbow Six Siege
- - 14. : Battlefield™ 6
- - 15. : skate.
- - 16. : Brick Rigs
- - 17. : Call of Duty®
- - 18. : Teardown
- - 19. : Call of Duty®: Modern Warfare®
- - 20. : BattleBit Remastered
- - 21. : Gorilla Tag
- - 22. : Wallpaper Engine
- - 23. : Crosshair V2
- - 24. : Counter-Strike 2
- - 25. : Ready or Not
- - 26. : Lethal Company
- - 27. : City Car Driving
- - 28. : Muck
- - 29. : Just Cause 3
- 0번 게임의 1번 유저의 보유 게임 30개 데이터 수집
- - 0. : Counter-Strike 2
- - 1. : Terraria
- - 2. : The Binding of Isaac: Rebirth
- - 3. : Rocket League
- - 4. : ELDEN RING
- - 5. : STAR WARS™ Battlefront™ II
- - 6